# v22b.1 Inference — All-label F1 + BoN N=3

Goal:
- Load Qwen3-8B + v22b LoRA
- Use Unsloth inference where possible
- BoN N=3
- Print **F1 for every label**: A/B/C/D/Yes/No/Unknown
- Apply strict gates for every label, not just macro-F1
- Save results/submission

This notebook does not use vLLM.

In [ ]:
# ==================================================================
# STAGE 0 -- Imports / install
# ==================================================================
import os, sys, subprocess, json, re, random, time, math, shutil, zipfile, glob
from pathlib import Path
from collections import Counter, defaultdict

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def _pip(*pkgs):
    print("pip install", " ".join(pkgs), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs], check=False)

try:
    import unsloth
except Exception:
    _pip("unsloth", "peft", "bitsandbytes", "accelerate", "safetensors", "scikit-learn")

import torch
from unsloth import FastLanguageModel
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, f1_score, classification_report, confusion_matrix

print("="*80)
print("IMPORT SUMMARY")
print("="*80)
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "CUDA=", torch.cuda.is_available(), "N_GPUS=", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ==================================================================
# STAGE 1 -- Config / path resolver
# ==================================================================
def resolve_path(candidates, label):
    hits = []
    for p in candidates:
        if any(ch in p for ch in "*?["):
            hits.extend(sorted(glob.glob(p, recursive=True)))
        else:
            hits.append(p)
    for p in hits:
        if os.path.exists(p):
            print(f"resolved {label}: {p}")
            return p
    print(f"WARNING: cannot resolve {label}; using first: {candidates[0]}")
    return candidates[0]

VERSION = "v22b_1"
SEED = 42
random.seed(SEED)

DATA_PATH = resolve_path([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "DATA v5")

TEST_PATH = resolve_path([
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/**/generated_v4style_300.json",
], "TEST generated_v4style_300")

QWEN_MODEL_ID = resolve_path([
    "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1",
    "/kaggle/input/**/qwen-3/transformers/8b/1",
    "/kaggle/input/**/Qwen3-8B*",
], "Qwen3-8B")

# Locate v22b LoRA zip or folder
COT_LORA_SRC = resolve_path([
    "/kaggle/input/**/qwen3_cot_lora_v22b_v5.zip",
    "/kaggle/input/**/qwen3_cot_lora_v22b_v5",
    "/kaggle/working/qwen3_cot_lora_v22b_v5.zip",
    "/kaggle/working/qwen3_cot_lora_v22b_v5",
], "v22b LoRA")

WORK_LORA_DIR = Path("/kaggle/working/qwen3_cot_lora_v22b_v5")
if str(COT_LORA_SRC).endswith(".zip"):
    if WORK_LORA_DIR.exists():
        shutil.rmtree(WORK_LORA_DIR)
    WORK_LORA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(COT_LORA_SRC, "r") as z:
        z.extractall(WORK_LORA_DIR)
    COT_LORA_PATH = str(WORK_LORA_DIR)
else:
    COT_LORA_PATH = COT_LORA_SRC

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True
QWEN_BON_N = 3
QWEN_BON_TEMP = 0.5
TOP_P = 0.95
MAX_NEW_TOKENS = 512
BATCH_SIZE = 4

OUT_JSON = "/kaggle/working/pipeline_results_22b_1_all_label_f1.json"
SUBMISSION_JSON = "/kaggle/working/submission_v22b_1.json"
SUBMISSION_CSV = "/kaggle/working/submission_v22b_1.csv"

V14_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

LABELS = ["A","B","C","D","Yes","No","Unknown"]

print("CONFIG OK")
print("DATA_PATH:", DATA_PATH)
print("TEST_PATH:", TEST_PATH)
print("QWEN_MODEL_ID:", QWEN_MODEL_ID)
print("COT_LORA_PATH:", COT_LORA_PATH)

In [ ]:
# ==================================================================
# STAGE 2 -- Load data, split by sample_id
# ==================================================================
from sklearn.model_selection import train_test_split

def norm_ans(x):
    u = str(x).strip().upper()
    if u in ("YES", "Y"): return "Yes"
    if u in ("NO", "N"): return "No"
    if u in ("UNKNOWN", "UNCERTAIN", "UNK"): return "Unknown"
    if u in ("A","B","C","D"): return u
    return str(x).strip()

def get_list_field(obj, names, default=None):
    for n in names:
        if n in obj:
            return obj[n]
    return default

def flatten_samples(samples, has_gold=True):
    flat = []
    for si, s in enumerate(samples):
        premises = get_list_field(s, ["premises-NL", "premises_NL", "premises"], [])
        qs = get_list_field(s, ["questions", "question"], [])
        ans = get_list_field(s, ["answers", "answer"], []) if has_gold else []
        exps = get_list_field(s, ["explanation", "explanations"], [])
        idxs = get_list_field(s, ["idx", "indices"], None)

        if isinstance(qs, str): qs = [qs]
        if has_gold and isinstance(ans, str): ans = [ans] * len(qs)
        if isinstance(exps, str): exps = [exps] * len(qs)
        if exps is None: exps = [""] * len(qs)

        for qi, q in enumerate(qs):
            item = {
                "sample_id": si,
                "q_idx": qi,
                "premises": premises,
                "question": q,
                "explanation_gold": exps[qi] if qi < len(exps) else "",
                "idx": idxs[qi] if isinstance(idxs, list) and qi < len(idxs) else None,
            }
            if has_gold:
                item["gold"] = norm_ans(ans[qi] if qi < len(ans) else "")
            flat.append(item)
    return flat

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data_samples = json.load(f)
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_samples = json.load(f)

records = flatten_samples(data_samples, has_gold=True)
test_records = flatten_samples(test_samples, has_gold=False)

sample_ids = sorted(set(x["sample_id"] for x in records))
train_ids, val_ids = train_test_split(sample_ids, test_size=0.20, random_state=SEED)
train_ids, val_ids = set(train_ids), set(val_ids)
train_recs = [x for x in records if x["sample_id"] in train_ids]
val_recs   = [x for x in records if x["sample_id"] in val_ids]

print("DATA records:", len(data_samples), "questions:", len(records))
print("TEST records:", len(test_samples), "questions:", len(test_records))
print("Train:", len(train_recs), Counter(x["gold"] for x in train_recs))
print("Val  :", len(val_recs), Counter(x["gold"] for x in val_recs))
print("Full :", len(records), Counter(x["gold"] for x in records))

In [ ]:
# ==================================================================
# STAGE 3 -- Prompt / extraction
# ==================================================================
ANSWER_RE = re.compile(r"Final\s*Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)", re.I)

def format_premises(premises):
    return "\n".join([f"Premise {i+1}: {p}" for i,p in enumerate(premises)])

def build_prompt(item):
    return (
        f"Premises:\n{format_premises(item['premises'])}\n\n"
        f"Question:\n{item['question']}\n\n"
        "Reason step-by-step by citing relevant premise numbers. "
        "End with exactly one line: Final Answer: X"
    )

def extract_answer(text):
    m = ANSWER_RE.search(text or "")
    if m:
        return norm_ans(m.group(1))
    # take last occurrence among labels, but avoid very early labels in MC options
    labels = []
    for m in re.finditer(r"\b(Yes|No|Unknown|A|B|C|D)\b", text or "", re.I):
        labels.append(norm_ans(m.group(1)))
    return labels[-1] if labels else "UNPARSEABLE"

def extract_cited_premises(text, n_premises=None):
    cited = set()
    patterns = [
        r"\b[Pp]remise\s*(\d+)\b",
        r"\b[Pp]remises\s*(\d+)\b",
        r"\bP\s*(\d+)\b",
    ]
    for pat in patterns:
        for m in re.finditer(pat, text or ""):
            try:
                v = int(m.group(1))
                if v >= 1 and (n_premises is None or v <= n_premises):
                    cited.add(v)
            except Exception:
                pass
    return sorted(cited)

print("Prompt/extraction OK")

In [ ]:
# ==================================================================
# STAGE 4 -- Load model + v22b LoRA via Unsloth
# ==================================================================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = QWEN_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

# Load adapter
from peft import PeftModel
model = PeftModel.from_pretrained(model, COT_LORA_PATH)
FastLanguageModel.for_inference(model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model + v22b LoRA loaded")

In [ ]:
# ==================================================================
# STAGE 5 -- Batch generation BoN N=3
# ==================================================================
def make_chat_prompt(item):
    messages = [
        {"role": "system", "content": V14_COT_SYSTEM},
        {"role": "user", "content": build_prompt(item)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(prompts, n=QWEN_BON_N, temperature=QWEN_BON_TEMP, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    all_outputs = []
    # We run each prompt n times by duplicating prompts, then regroup
    expanded = []
    for i,p in enumerate(prompts):
        for _ in range(n):
            expanded.append((i,p))
    results = [[] for _ in prompts]

    t0 = time.time()
    for start in range(0, len(expanded), batch_size):
        chunk = expanded[start:start+batch_size]
        idxs = [x[0] for x in chunk]
        texts = [x[1] for x in chunk]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        for j, ii in enumerate(idxs):
            gen = tokenizer.decode(out[j][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            results[ii].append(gen)
        if (start // batch_size) % 20 == 0:
            done = min(start+batch_size, len(expanded))
            print(f"  generated {done}/{len(expanded)} samples in {(time.time()-t0)/60:.1f} min", flush=True)
    return results

def predict_records(recs, title="eval"):
    print("="*70)
    print(f"PREDICT {title}: {len(recs)} questions, BoN={QWEN_BON_N}")
    print("="*70)
    prompts = [make_chat_prompt(x) for x in recs]
    bon_texts = generate_batch(prompts)
    out = []
    for item, texts in zip(recs, bon_texts):
        answers = [extract_answer(t) for t in texts]
        cnt = Counter(answers)
        # majority; deterministic tie order by count then label priority
        priority = {lab:i for i,lab in enumerate(LABELS + ["UNPARSEABLE"])}
        pred = sorted(cnt.items(), key=lambda kv: (-kv[1], priority.get(kv[0],99)))[0][0]
        # choose explanation from a sample with majority answer
        chosen = next((t for t,a in zip(texts,answers) if a == pred), texts[0] if texts else "")
        cited = extract_cited_premises(chosen, n_premises=len(item["premises"]))
        rec = dict(item)
        rec.update({
            "pred": pred,
            "bon_answers": answers,
            "bon_counts": dict(cnt),
            "explanation": chosen,
            "cited_premises": cited,
        })
        out.append(rec)
    return out

print("Generation functions OK")

In [ ]:
# ==================================================================
# STAGE 6 -- Metrics: all-label F1 + recall gates
# ==================================================================
F1_GATES = {
    "A": 0.55,
    "B": 0.50,   # revised: stricter than old 0.45
    "C": 0.40,   # revised: stricter than old 0.35
    "D": 0.35,
    "Yes": 0.60,
    "No": 0.60,
    "Unknown": 0.55,
}
RECALL_GATE = 0.30
MC_MACRO_F1_GATE = 0.65
MACRO_F1_GATE = 0.60

def compute_metrics(pred_recs, name="VAL"):
    y_true = [r["gold"] for r in pred_recs]
    y_pred = [r["pred"] for r in pred_recs]

    labels = LABELS
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    per = {
        lab: {
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1": float(f1[i]),
            "support": int(sup[i]),
            "pred_count": int(sum(1 for x in y_pred if x == lab)),
        }
        for i, lab in enumerate(labels)
    }
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    weighted = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

    mc_labels = ["A","B","C","D"]
    ynu_labels = ["Yes","No","Unknown"]
    mc_mask = [yt in mc_labels for yt in y_true]
    ynu_mask = [yt in ynu_labels for yt in y_true]

    def sub_metric(mask, labs):
        yt = [a for a, m in zip(y_true, mask) if m]
        yp = [a for a, m in zip(y_pred, mask) if m]
        if not yt:
            return {"acc": None, "macro_f1": None, "n": 0}
        return {
            "acc": float(accuracy_score(yt, yp)),
            "macro_f1": float(f1_score(yt, yp, labels=labs, average="macro", zero_division=0)),
            "n": len(yt),
        }

    metrics = {
        "name": name,
        "n": len(pred_recs),
        "acc": float(acc),
        "macro_f1": float(macro),
        "weighted_f1": float(weighted),
        "per_label": per,
        "mc": sub_metric(mc_mask, mc_labels),
        "ynu": sub_metric(ynu_mask, ynu_labels),
        "pred_dist": dict(Counter(y_pred)),
        "gold_dist": dict(Counter(y_true)),
        "citation": {
            "with_citation": int(sum(1 for rr in pred_recs if rr.get("cited_premises"))),
            "avg_cited": float(sum(len(rr.get("cited_premises", [])) for rr in pred_recs) / max(1, len(pred_recs))),
            "citation_rate": float(sum(1 for rr in pred_recs if rr.get("cited_premises")) / max(1, len(pred_recs))),
        }
    }
    return metrics

def print_metrics(m):
    print("\n" + "="*100)
    print(f"{m['name']} METRICS -- ALL LABEL F1 + RECALL GATES")
    print("="*100)
    print(f"N={m['n']}  acc={m['acc']:.3f}  macro_f1={m['macro_f1']:.3f}  weighted_f1={m['weighted_f1']:.3f}")
    print(f"MC:  acc={m['mc']['acc']:.3f}  macro_f1={m['mc']['macro_f1']:.3f}  n={m['mc']['n']}")
    print(f"YNU: acc={m['ynu']['acc']:.3f}  macro_f1={m['ynu']['macro_f1']:.3f}  n={m['ynu']['n']}")
    print("-"*100)
    print(f"{'Label':10s} {'P':>8s} {'R':>8s} {'F1':>8s} {'Gold':>8s} {'Pred':>8s} {'F1Gate':>8s} {'RGate':>8s}")
    for lab in LABELS:
        x = m["per_label"][lab]
        f_ok = x["f1"] >= F1_GATES[lab]
        r_ok = x["recall"] >= RECALL_GATE
        print(
            f"{lab:10s} {x['precision']:8.3f} {x['recall']:8.3f} {x['f1']:8.3f} "
            f"{x['support']:8d} {x['pred_count']:8d} "
            f"{'PASS' if f_ok else 'FAIL':>8s} {'PASS' if r_ok else 'FAIL':>8s}"
        )
    print("-"*100)
    print("Gold dist:", m["gold_dist"])
    print("Pred dist:", m["pred_dist"])
    print("Citation:", m["citation"])

def gate_status(m):
    per = m["per_label"]
    gate = {}
    fail_detail = []

    for lab, tau in F1_GATES.items():
        key = f"{lab}_F1>={tau:.2f}"
        ok = per[lab]["f1"] >= tau
        gate[key] = ok
        if not ok:
            fail_detail.append({
                "label": lab,
                "type": "f1",
                "value": per[lab]["f1"],
                "threshold": tau,
                "precision": per[lab]["precision"],
                "recall": per[lab]["recall"],
                "support": per[lab]["support"],
                "pred_count": per[lab]["pred_count"],
            })

    for lab in LABELS:
        key = f"{lab}_recall>={RECALL_GATE:.2f}"
        ok = per[lab]["recall"] >= RECALL_GATE
        gate[key] = ok
        if not ok:
            fail_detail.append({
                "label": lab,
                "type": "recall",
                "value": per[lab]["recall"],
                "threshold": RECALL_GATE,
                "precision": per[lab]["precision"],
                "f1": per[lab]["f1"],
                "support": per[lab]["support"],
                "pred_count": per[lab]["pred_count"],
            })

    gate[f"MC_macro_F1>={MC_MACRO_F1_GATE:.2f}"] = (m["mc"]["macro_f1"] is not None and m["mc"]["macro_f1"] >= MC_MACRO_F1_GATE)
    gate[f"Macro_F1>={MACRO_F1_GATE:.2f}"] = (m["macro_f1"] >= MACRO_F1_GATE)

    if not gate[f"MC_macro_F1>={MC_MACRO_F1_GATE:.2f}"]:
        fail_detail.append({"label": "MC", "type": "macro_f1", "value": m["mc"]["macro_f1"], "threshold": MC_MACRO_F1_GATE})
    if not gate[f"Macro_F1>={MACRO_F1_GATE:.2f}"]:
        fail_detail.append({"label": "ALL", "type": "macro_f1", "value": m["macro_f1"], "threshold": MACRO_F1_GATE})

    print("\n" + "="*80)
    print(f"{m['name']} GATE STATUS -- ALL LABEL F1 + RECALL")
    print("="*80)
    for k, v in gate.items():
        print(f"{k:26s}: {'PASS' if v else 'FAIL'}")
    print("-"*80)
    if fail_detail:
        print("FAIL DETAIL:")
        for d in fail_detail:
            print(" ", d)
    else:
        print("All gates passed.")
    print("OVERALL:", "PASS" if all(gate.values()) else "FAIL")

    return gate, fail_detail

print("Metrics functions OK: revised gates B=0.50, C=0.40, recall gate=0.30")


In [ ]:
# ==================================================================
# STAGE 7 -- Run VAL eval first
# ==================================================================
t0 = time.time()
val_pred = predict_records(val_recs, "VAL")
print("VAL predict minutes:", (time.time()-t0)/60)
val_metrics = compute_metrics(val_pred, "VAL")
print_metrics(val_metrics)
val_gate = gate_status(val_metrics)

In [ ]:
# ==================================================================
# STAGE 8 -- Optional FULL + TEST inference
# ==================================================================
RUN_FULL = True
RUN_TEST = True

all_outputs = {"version": VERSION, "config": {
    "bon_n": QWEN_BON_N, "temp": QWEN_BON_TEMP, "top_p": TOP_P, "lora": COT_LORA_PATH,
}}

if RUN_FULL:
    t0 = time.time()
    full_pred = predict_records(records, "FULL")
    print("FULL predict minutes:", (time.time()-t0)/60)
    full_metrics = compute_metrics(full_pred, "FULL")
    print_metrics(full_metrics)
    full_gate = gate_status(full_metrics)
    all_outputs["full_metrics"] = full_metrics
    all_outputs["full_predictions"] = full_pred

if RUN_TEST:
    t0 = time.time()
    test_pred = predict_records(test_records, "TEST")
    print("TEST predict minutes:", (time.time()-t0)/60)
    # Print submission distribution
    cnt = Counter(r["pred"] for r in test_pred)
    print("\n" + "="*70)
    print("TEST SUBMISSION DISTRIBUTION")
    print("="*70)
    for lab in LABELS + ["UNPARSEABLE"]:
        print(f"{lab:12s}: {cnt.get(lab,0)}")
    print("Total:", sum(cnt.values()))
    print("\nFirst 10 test predictions:")
    for r in test_pred[:10]:
        print(f"sample={r['sample_id']} q={r['q_idx']} pred={r['pred']} cited={r['cited_premises']} counts={r['bon_counts']}")
    all_outputs["test_predictions"] = test_pred
    all_outputs["test_distribution"] = dict(cnt)

all_outputs["val_metrics"] = val_metrics
all_outputs["val_predictions"] = val_pred

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)

# Submission files
if RUN_TEST:
    submission = []
    for r in test_pred:
        submission.append({
            "sample_id": r["sample_id"],
            "q_idx": r["q_idx"],
            "answer": r["pred"],
            "explanation": r.get("explanation",""),
            "supporting_premises": r.get("cited_premises", []),
        })
    with open(SUBMISSION_JSON, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    import csv
    with open(SUBMISSION_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["sample_id","q_idx","answer","explanation","supporting_premises"])
        w.writeheader()
        for row in submission:
            row = dict(row)
            row["supporting_premises"] = json.dumps(row["supporting_premises"])
            w.writerow(row)

print("\nSaved:", OUT_JSON)
print("Saved:", SUBMISSION_JSON)
print("Saved:", SUBMISSION_CSV)
print("v22b.1 inference completed")